# 4-1 비선형성 심화

강의 원문 대신 직접 작성하고 실행한 코드와 학습 메모를 정리했습니다.


In [6]:
import torch
import torch.nn as nn

x = torch.tensor([[1., 2.], [-1., 0.5]])
W1 = torch.tensor([[1., 2.], [0., -1.]])
b1 = torch.tensor([0.5, -0.5])
W2 = torch.tensor([[2., -1.]])
b2 = torch.tensor([0.25])

two_layer = (x @ W1.T + b1) @ W2.T + b2
two_layer = x @ W1.T @ W2.T + b1 @ W2.T +b2

W_new = W2 @ W1
b_new = b1 @ W2.T + b2
one_layer = x @ W_new.T + b_new

print(two_layer)
print(one_layer)

tensor([[13.7500],
        [ 2.2500]])
tensor([[13.7500],
        [ 2.2500]])


In [7]:
# 검증 가능 정답 코드
import torch
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([0, 1, 1, 0])
W = torch.tensor([[1., -1.], [-1., 1.]])
hidden = torch.relu(X @ W.T)  # 서로 다른 두 방향의 차이를 음수 없이 보존합니다.
# 두 hidden 반응을 합친 뒤 0.5 경계를 적용하면 선형 한 층으로는 만들 수 없던 XOR 경계를 얻습니다.
logits = hidden.sum(dim=1) - 0.5
preds = (logits >= 0).long()
print("hidden:", hidden.tolist())
print("preds:", preds.tolist())
print("matches:", torch.equal(preds, y))

hidden: [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 0.0]]
preds: [0, 1, 1, 0]
matches: True


In [8]:
# 검증 가능 정답 코드
candidates = {
    "L": {"xor_acc": 0.75, "valid_acc": 0.72, "params": 3, "latency": 0.2},
    "R": {"xor_acc": 1.00, "valid_acc": 0.86, "params": 9, "latency": 0.4},
    "W": {"xor_acc": 1.00, "valid_acc": 0.91, "params": 41, "latency": 1.2},
}
# XOR 단위 검증, validation 품질, 파라미터·지연 예산을 서로 독립된 hard contract로 검사합니다.
checks = {
    name: {
        "xor": result["xor_acc"] == 1.0,
        "validation": result["valid_acc"] >= 0.85,
        "budget": result["params"] <= 20 and result["latency"] <= 0.8,
    }
    for name, result in candidates.items()
}
# 모든 계약을 통과한 후보 안에서만 validation accuracy를 비교해 큰 모델의 예산 위반을 숨기지 않습니다.
eligible = [name for name, result in checks.items() if all(result.values())]
selected = max(eligible, key=lambda name: candidates[name]["valid_acc"]) if eligible else "보류"
print("checks:", checks)
print("selected:", selected)

checks: {'L': {'xor': False, 'validation': False, 'budget': True}, 'R': {'xor': True, 'validation': True, 'budget': True}, 'W': {'xor': True, 'validation': True, 'budget': False}}
selected: R
